In [1]:
from pyspark.sql import SparkSession

spark =(
    SparkSession.builder
    .appName("ExoticindianEatsAnalytics")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/14 22:51:07 WARN Utils: Your hostname, BHANUs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.220 instead (on interface en0)
26/07/14 22:51:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/14 22:51:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


In [2]:
spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS")

'hdfs://localhost:9000'

In [3]:
bronze_base = "hdfs://localhost:9000/user/bhanupendyala/exotic-indian-eats/bronze"

orders_path = f"{bronze_base}/orders.csv"
order_items_path = f"{bronze_base}/order_items.csv"
products_path = f"{bronze_base}/products.csv"
customers_path = f"{bronze_base}/customers.csv"
payments_path = f"{bronze_base}/payments.csv"
promotions_path = f"{bronze_base}/promotions.csv"
reviews_path = f"{bronze_base}/reviews.csv"

In [4]:
orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(orders_path)
)

In [5]:
order_items_df = spark.read.option("header", True).option("inferSchema", True).csv(order_items_path)

In [6]:
products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(products_path)
)

customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(customers_path)
)


payments_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(payments_path)
)

promotions_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(promotions_path)
)

reviews_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(reviews_path)
)


In [7]:
orders_df.show(5)
orders_df.printSchema()

+--------+----------+-------------------+-------------------+---------+-----------+------------+------------+--------+--------------+-----+-----+------------+-------------------+------------+------------+-------------------+-------------------+-----------+
|order_id|order_date|         order_time|     order_datetime| platform|customer_id|order_status|promotion_id|subtotal|order_discount|  tax|  tip|delivery_fee|platform_commission|total_amount|prep_minutes|     ready_datetime| fulfilled_datetime|server_name|
+--------+----------+-------------------+-------------------+---------+-----------+------------+------------+--------+--------------+-----+-----+------------+-------------------+------------+------------+-------------------+-------------------+-----------+
|O0000001|2025-07-01|2026-07-14 11:55:00|2025-07-01 11:55:00|Uber Eats|     C00886|   Completed|        NULL|   21.99|           0.0| 2.86|  0.0|        3.01|               6.16|       27.86|          22|2025-07-01 12:17:00|2025-

In [8]:
print("Orders:", orders_df.count())
print("Order items:", order_items_df.count())
print("Products:", products_df.count())
print("Customers:", customers_df.count())
print("Payments:", payments_df.count())
print("Promotions:", promotions_df.count())
print("Reviews:", reviews_df.count())

Orders: 14101
Order items: 43917
Products: 40
Customers: 2500
Payments: 14101
Promotions: 7
Reviews: 2105


In [9]:
from pyspark.sql import functions as F

In [10]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_time: timestamp (nullable = true)
 |-- order_datetime: timestamp (nullable = true)
 |-- platform: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- promotion_id: string (nullable = true)
 |-- subtotal: double (nullable = true)
 |-- order_discount: double (nullable = true)
 |-- tax: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- delivery_fee: double (nullable = true)
 |-- platform_commission: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- prep_minutes: integer (nullable = true)
 |-- ready_datetime: timestamp (nullable = true)
 |-- fulfilled_datetime: timestamp (nullable = true)
 |-- server_name: string (nullable = true)



In [11]:
orders_clean_df = (
    orders_df
    .dropDuplicates(["order_id"])
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("order_datetime", F.to_timestamp("order_datetime"))
    .withColumn("ready_datetime", F.to_timestamp("ready_datetime"))
    .withColumn("fulfilled_datetime", F.to_timestamp("fulfilled_datetime"))
    .withColumn("subtotal", F.col("subtotal").cast("double"))
    .withColumn("order_discount", F.col("order_discount").cast("double"))
    .withColumn("tax", F.col("tax").cast("double"))
    .withColumn("tip", F.col("tip").cast("double"))
    .withColumn("delivery_fee", F.col("delivery_fee").cast("double"))
    .withColumn("platform_commission", F.col("platform_commission").cast("double"))
    .withColumn("total_amount", F.col("total_amount").cast("double"))
    .withColumn("prep_minutes", F.col("prep_minutes").cast("integer"))
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("day_name", F.date_format("order_date", "EEEE"))
)

In [12]:
orders_clean_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in orders_clean_df.columns
]).show(vertical=True)

26/07/14 22:51:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


-RECORD 0--------------------
 order_id            | 0     
 order_date          | 0     
 order_time          | 0     
 order_datetime      | 0     
 platform            | 0     
 customer_id         | 2005  
 order_status        | 0     
 promotion_id        | 11532 
 subtotal            | 0     
 order_discount      | 0     
 tax                 | 0     
 tip                 | 0     
 delivery_fee        | 0     
 platform_commission | 0     
 total_amount        | 0     
 prep_minutes        | 0     
 ready_datetime      | 0     
 fulfilled_datetime  | 0     
 server_name         | 9716  
 order_year          | 0     
 order_month         | 0     
 day_name            | 0     



In [13]:
orders_clean_df = orders_clean_df.filter(
    F.col("order_status").isin("Completed", "Cancelled", "Refunded")
)

In [14]:
orders_clean_df.select("order_id","platform","order_status").show(5,vertical= False)

+--------+---------+------------+
|order_id| platform|order_status|
+--------+---------+------------+
|O0000001|Uber Eats|   Completed|
|O0000002|  Dine-In|   Completed|
|O0000003|  Dine-In|   Completed|
|O0000004|  Dine-In|   Completed|
|O0000005|   Pickup|   Completed|
+--------+---------+------------+
only showing top 5 rows


In [15]:
order_items_clean_df = (
    order_items_df
    .dropDuplicates(["order_item_id"])
    .withColumn("quantity", F.col("quantity").cast("integer"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("line_discount", F.col("line_discount").cast("double"))
    .withColumn("line_total", F.col("line_total").cast("double"))
    .filter(F.col("quantity") > 0)
)

In [16]:
products_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- diet_type: string (nullable = true)
 |-- selling_price: double (nullable = true)
 |-- estimated_food_cost: double (nullable = true)



In [17]:
products_clean_df = (
    products_df
    .dropDuplicates(["product_id"])
    .withColumn("selling_price", F.col("selling_price").cast("double"))
    .withColumn(
        "estimated_food_cost",
        F.col("estimated_food_cost").cast("double")
    )
    .withColumn(
        "estimated_profit_per_item",
        F.col("selling_price") - F.col("estimated_food_cost")
    )
)

In [18]:
products_clean_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- diet_type: string (nullable = true)
 |-- selling_price: double (nullable = true)
 |-- estimated_food_cost: double (nullable = true)
 |-- estimated_profit_per_item: double (nullable = true)



In [19]:
products_clean_df.show(5)

+----------+--------------------+--------+---------+-------------+-------------------+-------------------------+
|product_id|        product_name|category|diet_type|selling_price|estimated_food_cost|estimated_profit_per_item|
+----------+--------------------+--------+---------+-------------+-------------------+-------------------------+
|      P001|      Butter Chicken|   Curry|  Non-Veg|        18.99|                6.2|                    12.79|
|      P002|Chicken Tikka Masala|   Curry|  Non-Veg|        19.49|                6.5|       12.989999999999998|
|      P003|       Kadai Chicken|   Curry|  Non-Veg|        19.99|                6.8|       13.189999999999998|
|      P004|  Chicken Ghee Roast|   Curry|  Non-Veg|        20.99|                7.3|       13.689999999999998|
|      P005|          Goat Curry|   Curry|  Non-Veg|        21.99|                8.9|       13.089999999999998|
+----------+--------------------+--------+---------+-------------+-------------------+----------

In [20]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Exotic Indian Eats") \
    .master("local[*]") \
    .getOrCreate()

26/07/14 22:51:52 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [21]:
products_clean_df.show(5)

+----------+--------------------+--------+---------+-------------+-------------------+-------------------------+
|product_id|        product_name|category|diet_type|selling_price|estimated_food_cost|estimated_profit_per_item|
+----------+--------------------+--------+---------+-------------+-------------------+-------------------------+
|      P001|      Butter Chicken|   Curry|  Non-Veg|        18.99|                6.2|                    12.79|
|      P002|Chicken Tikka Masala|   Curry|  Non-Veg|        19.49|                6.5|       12.989999999999998|
|      P003|       Kadai Chicken|   Curry|  Non-Veg|        19.99|                6.8|       13.189999999999998|
|      P004|  Chicken Ghee Roast|   Curry|  Non-Veg|        20.99|                7.3|       13.689999999999998|
|      P005|          Goat Curry|   Curry|  Non-Veg|        21.99|                8.9|       13.089999999999998|
+----------+--------------------+--------+---------+-------------+-------------------+----------

In [23]:
products_clean_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- diet_type: string (nullable = true)
 |-- selling_price: double (nullable = true)
 |-- estimated_food_cost: double (nullable = true)
 |-- estimated_profit_per_item: double (nullable = true)



In [24]:
silver_base = "hdfs://localhost:9000/user/bhanupendyala/exotic-indian-eats/silver"

In [25]:
(
    orders_clean_df.write
    .mode("overwrite")
    .partitionBy("order_year", "order_month")
    .parquet(f"{silver_base}/orders")
)

(
    order_items_clean_df.write
    .mode("overwrite")
    .parquet(f"{silver_base}/order_items")
)

(
    products_clean_df.write
    .mode("overwrite")
    .parquet(f"{silver_base}/products")
)

(
    customers_df.write
    .mode("overwrite")
    .parquet(f"{silver_base}/customers")
)

(
    payments_df.write
    .mode("overwrite")
    .parquet(f"{silver_base}/payments")
)

(
    promotions_df.write
    .mode("overwrite")
    .parquet(f"{silver_base}/promotions")
)

(
    reviews_df.write
    .mode("overwrite")
    .parquet(f"{silver_base}/reviews")
)

In [26]:
silver_orders_df = spark.read.parquet(f"{silver_base}/orders")
silver_order_items_df = spark.read.parquet(f"{silver_base}/order_items")
silver_products_df = spark.read.parquet(f"{silver_base}/products")

In [27]:
silver_orders_df.show(5)

+--------+----------+-------------------+-------------------+-------------+-----------+------------+------------+--------+--------------+-----+----+------------+-------------------+------------+------------+-------------------+-------------------+-----------+--------+----------+-----------+
|order_id|order_date|         order_time|     order_datetime|     platform|customer_id|order_status|promotion_id|subtotal|order_discount|  tax| tip|delivery_fee|platform_commission|total_amount|prep_minutes|     ready_datetime| fulfilled_datetime|server_name|day_name|order_year|order_month|
+--------+----------+-------------------+-------------------+-------------+-----------+------------+------------+--------+--------------+-----+----+------------+-------------------+------------+------------+-------------------+-------------------+-----------+--------+----------+-----------+
|O0004756|2025-11-01|2026-07-14 13:05:00|2025-11-01 13:05:00|      Dine-In|       NULL|   Completed|       PR006|   28.48|  

In [29]:
silver_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_time: timestamp (nullable = true)
 |-- order_datetime: timestamp (nullable = true)
 |-- platform: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- promotion_id: string (nullable = true)
 |-- subtotal: double (nullable = true)
 |-- order_discount: double (nullable = true)
 |-- tax: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- delivery_fee: double (nullable = true)
 |-- platform_commission: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- prep_minutes: integer (nullable = true)
 |-- ready_datetime: timestamp (nullable = true)
 |-- fulfilled_datetime: timestamp (nullable = true)
 |-- server_name: string (nullable = true)
 |-- day_name: string (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)



In [30]:
silver_orders_df.createOrReplaceTempView("orders")
silver_order_items_df.createOrReplaceTempView("order_items")
silver_products_df.createOrReplaceTempView("products")
customers_df.createOrReplaceTempView("customers")
reviews_df.createOrReplaceTempView("reviews")

In [35]:
spark.sql("""
SELECT
    ROUND(SUM(total_amount), 2) AS total_sales
FROM orders
WHERE order_status = 'Completed'
""").show()

+-----------+
|total_sales|
+-----------+
| 1037730.77|
+-----------+



In [36]:
monthly_sales_df = spark.sql("""
SELECT
    DATE_FORMAT(order_date, 'yyyy-MM') AS sales_month,
    COUNT(*) AS order_count,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(total_amount), 2) AS average_order_value
FROM orders
WHERE order_status = 'Completed'
GROUP BY DATE_FORMAT(order_date, 'yyyy-MM')
ORDER BY sales_month
""")

monthly_sales_df.show()

+-----------+-----------+-------------+-------------------+
|sales_month|order_count|total_revenue|average_order_value|
+-----------+-----------+-------------+-------------------+
|    2025-07|       1128|     87140.92|              77.25|
|    2025-08|       1161|     90679.11|               78.1|
|    2025-09|       1062|     82919.99|              78.08|
|    2025-10|       1188|     88323.07|              74.35|
|    2025-11|       1293|     98407.51|              76.11|
|    2025-12|       1166|     90077.24|              77.25|
|    2026-01|       1107|     87626.69|              79.16|
|    2026-02|        945|     72606.94|              76.83|
|    2026-03|       1139|     87770.49|              77.06|
|    2026-04|       1020|     78405.86|              76.87|
|    2026-05|       1129|     86870.81|              76.94|
|    2026-06|       1128|     86902.14|              77.04|
+-----------+-----------+-------------+-------------------+



In [37]:
platform_sales_df = spark.sql("""
SELECT
    platform,
    COUNT(*) AS order_count,
    ROUND(SUM(total_amount), 2) AS gross_sales,
    ROUND(SUM(platform_commission), 2) AS total_commission,
    ROUND(
        SUM(total_amount) - SUM(platform_commission),
        2
    ) AS estimated_net_sales
FROM orders
WHERE order_status = 'Completed'
GROUP BY platform
ORDER BY gross_sales DESC
""")

platform_sales_df.show()

+-------------+-----------+-----------+----------------+-------------------+
|     platform|order_count|gross_sales|total_commission|estimated_net_sales|
+-------------+-----------+-----------+----------------+-------------------+
|      Dine-In|       4173|  334940.51|             0.0|          334940.51|
|    Uber Eats|       3609|  274616.36|        63836.95|          210779.41|
|       Pickup|       3236|  234705.85|             0.0|          234705.85|
|SkipTheDishes|       2448|  193468.05|        41855.04|          151613.01|
+-------------+-----------+-----------+----------------+-------------------+



In [41]:
top_items_df = spark.sql("""
SELECT
    product_name,
    category,
    SUM(quantity) AS quantity_sold,
    ROUND(SUM(line_total), 2) AS item_revenue
FROM order_items
GROUP BY product_name, category
ORDER BY quantity_sold DESC
LIMIT 5
""")

top_items_df.show(truncate=False)

+-------------------------+--------+-------------+------------+
|product_name             |category|quantity_sold|item_revenue|
+-------------------------+--------+-------------+------------+
|Goat Biryani             |Biryani |2279         |49591.2     |
|Ulavacharu Mutton Biryani|Biryani |2224         |50562.16    |
|Veg Biryani              |Biryani |2217         |37233.67    |
|Special Boneless Biryani |Biryani |2215         |43781.47    |
|Fry Piece Chicken Biryani|Biryani |2194         |44470.04    |
+-------------------------+--------+-------------+------------+



In [40]:
top_items_df = spark.sql("""
SELECT * from order_items LIMIT 10
""")

top_items_df.show(truncate=False)

+-------------+--------+----------+----------------------+---------+--------+----------+-------------+----------+
|order_item_id|order_id|product_id|product_name          |category |quantity|unit_price|line_discount|line_total|
+-------------+--------+----------+----------------------+---------+--------+----------+-------------+----------+
|OI00000001   |O0000001|P005      |Goat Curry            |Curry    |1       |21.99     |0.0          |21.99     |
|OI00000002   |O0000002|P011      |Chana Masala          |Curry    |2       |14.99     |0.0          |29.98     |
|OI00000003   |O0000002|P034      |Jeera Rice            |Rice     |1       |5.49      |0.0          |5.49      |
|OI00000004   |O0000002|P014      |Goat Biryani          |Biryani  |2       |21.99     |0.0          |43.98     |
|OI00000005   |O0000002|P035      |Mango Lassi           |Beverage |1       |5.99      |0.0          |5.99      |
|OI00000006   |O0000003|P016      |Gongura Mutton Biryani|Biryani  |1       |22.99     |

In [63]:
category_sales_df = spark.sql("""
SELECT
    Category,
    sum(quantity) AS quantity_sold,
    ROUND(SUM(line_total), 2) AS revenue_by_category
FROM order_items
GROUP BY category
ORDER BY revenue_by_category DESC
LIMIT 10
""")
category_sales_df.show(truncate=False)


+---------+-------------+-------------------+
|Category |quantity_sold|revenue_by_category|
+---------+-------------+-------------------+
|Biryani  |17618        |367762.45          |
|Curry    |17833        |341028.77          |
|Appetizer|7671         |115091.97          |
|Dosa     |2541         |34363.89           |
|Bread    |5375         |23288.99           |
|Beverage |2473         |12994.44           |
|Dessert  |1653         |10312.27           |
|Rice     |235          |1275.74            |
+---------+-------------+-------------------+



In [49]:
peak_hours_df = spark.sql("""
SELECT
    HOUR(order_datetime) AS order_hour,
    COUNT(*) AS order_count,
    ROUND(SUM(total_amount), 2) AS revenue
FROM orders
WHERE order_status = 'Completed'
GROUP BY HOUR(order_datetime)
ORDER BY order_count DESC
""")

peak_hours_df.show()

+----------+-----------+---------+
|order_hour|order_count|  revenue|
+----------+-----------+---------+
|        20|       1372|107247.01|
|        21|       1370|106221.66|
|        19|       1367|105920.48|
|        18|       1355|106732.49|
|        16|       1340|101923.04|
|        17|       1337|100434.78|
|        22|       1273| 97498.55|
|        13|       1025|  75790.0|
|        12|       1017| 80451.92|
|        11|       1016| 78331.39|
|        14|        994| 77179.45|
+----------+-----------+---------+



In [50]:
day_sales_df = spark.sql("""
SELECT
    day_name,
    COUNT(*) AS order_count,
    ROUND(SUM(total_amount), 2) AS revenue
FROM orders
WHERE order_status = 'Completed'
GROUP BY day_name
ORDER BY revenue DESC
""")

day_sales_df.show()

+--------+-----------+---------+
|day_name|order_count|  revenue|
+--------+-----------+---------+
|Saturday|       2808|214555.85|
|  Friday|       2722|207815.95|
|  Sunday|       2349|183036.58|
| Tuesday|       1887|148836.01|
|  Monday|       1865|144504.03|
|Thursday|       1835|138982.35|
+--------+-----------+---------+



In [51]:
spark.sql("""
SELECT
    platform,
    ROUND(AVG(prep_minutes), 2) AS average_prep_minutes
FROM orders
WHERE order_status = 'Completed'
GROUP BY platform
ORDER BY average_prep_minutes DESC
""").show()

+-------------+--------------------+
|     platform|average_prep_minutes|
+-------------+--------------------+
|      Dine-In|               28.54|
|       Pickup|               23.74|
|    Uber Eats|               23.67|
|SkipTheDishes|               23.59|
+-------------+--------------------+



In [52]:
spark.sql("""
SELECT
    order_status,
    COUNT(*) AS order_count,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM orders
GROUP BY order_status
ORDER BY order_count DESC
""").show()

+------------+-----------+----------+
|order_status|order_count|percentage|
+------------+-----------+----------+
|   Completed|      13466|     95.50|
|   Cancelled|        406|      2.88|
|    Refunded|        229|      1.62|
+------------+-----------+----------+



26/07/15 00:28:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 00:28:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 00:28:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 00:28:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 00:28:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 00:28:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 0

In [53]:
item_profit_df = (
    silver_order_items_df.alias("oi")
    .join(
        silver_products_df.alias("p"),
        F.col("oi.product_id") == F.col("p.product_id"),
        "left"
    )
    .withColumn(
        "estimated_total_food_cost",
        F.col("oi.quantity") * F.col("p.estimated_food_cost")
    )
    .withColumn(
        "estimated_gross_profit",
        F.col("oi.line_total") - F.col("estimated_total_food_cost")
    )
)

In [55]:
item_profit_df.show(5)

+-------------+--------+----------+------------+--------+--------+----------+-------------+----------+----------+------------+--------+---------+-------------+-------------------+-------------------------+-------------------------+----------------------+
|order_item_id|order_id|product_id|product_name|category|quantity|unit_price|line_discount|line_total|product_id|product_name|category|diet_type|selling_price|estimated_food_cost|estimated_profit_per_item|estimated_total_food_cost|estimated_gross_profit|
+-------------+--------+----------+------------+--------+--------+----------+-------------+----------+----------+------------+--------+---------+-------------+-------------------+-------------------------+-------------------------+----------------------+
|   OI00000001|O0000001|      P005|  Goat Curry|   Curry|       1|     21.99|          0.0|     21.99|      P005|  Goat Curry|   Curry|  Non-Veg|        21.99|                8.9|       13.089999999999998|                      8.9|    

In [58]:
item_profit_df.select(
    F.col("oi.product_id"),
    F.col("oi.order_id"),
    F.col("line_total")
).show()


+----------+--------+----------+
|product_id|order_id|line_total|
+----------+--------+----------+
|      P005|O0000001|     21.99|
|      P011|O0000002|     29.98|
|      P034|O0000002|      5.49|
|      P014|O0000002|     43.98|
|      P035|O0000002|      5.99|
|      P016|O0000003|     22.99|
|      P032|O0000004|      5.99|
|      P030|O0000004|      3.99|
|      P032|O0000004|      5.99|
|      P021|O0000004|     16.49|
|      P016|O0000004|     68.97|
|      P019|O0000004|     33.98|
|      P031|O0000005|      4.49|
|      P019|O0000005|     16.99|
|      P022|O0000005|     35.98|
|      P001|O0000005|     18.99|
|      P002|O0000006|     38.98|
|      P015|O0000006|     21.49|
|      P008|O0000006|     17.99|
|      P007|O0000007|     39.58|
+----------+--------+----------+
only showing top 20 rows


In [59]:
product_profit_df = (
    item_profit_df
    .groupBy(
        F.col("oi.product_id").alias("product_id"),
        F.col("oi.product_name").alias("product_name"),
        F.col("oi.category").alias("category")
    )
    .agg(
        F.sum("oi.quantity").alias("quantity_sold"),
        F.round(F.sum("oi.line_total"), 2).alias("revenue"),
        F.round(
            F.sum("estimated_total_food_cost"),
            2
        ).alias("estimated_food_cost"),
        F.round(
            F.sum("estimated_gross_profit"),
            2
        ).alias("estimated_gross_profit")
    )
    .orderBy(F.desc("estimated_gross_profit"))
)

product_profit_df.show(10, truncate=False)

+----------+--------------------------+--------+-------------+--------+-------------------+----------------------+
|product_id|product_name              |category|quantity_sold|revenue |estimated_food_cost|estimated_gross_profit|
+----------+--------------------------+--------+-------------+--------+-------------------+----------------------+
|P014      |Goat Biryani              |Biryani |2279         |49591.2 |19599.4            |29991.8               |
|P018      |Ulavacharu Mutton Biryani |Biryani |2224         |50562.16|20683.2            |29878.96              |
|P017      |Ulavacharu Chicken Biryani|Biryani |2139         |46507.75|16684.2            |29823.55              |
|P016      |Gongura Mutton Biryani    |Biryani |2191         |49818.63|20157.2            |29661.43              |
|P015      |Gongura Chicken Biryani   |Biryani |2159         |45797.53|16408.4            |29389.13              |
|P013      |Fry Piece Chicken Biryani |Biryani |2194         |44470.04|15358.0  

In [60]:
gold_base = "hdfs://localhost:9000/user/bhanupendyala/exotic-indian-eats/gold"

In [64]:
monthly_sales_df.write.mode("overwrite").parquet(
    f"{gold_base}/monthly_sales"
)

platform_sales_df.write.mode("overwrite").parquet(
    f"{gold_base}/platform_sales"
)

top_items_df.write.mode("overwrite").parquet(
    f"{gold_base}/top_items"
)

category_sales_df.write.mode("overwrite").parquet(
    f"{gold_base}/category_sales_df"
)

peak_hours_df.write.mode("overwrite").parquet(
    f"{gold_base}/peak_hours"
)

day_sales_df.write.mode("overwrite").parquet(
    f"{gold_base}/day_sales"
)

product_profit_df.write.mode("overwrite").parquet(
    f"{gold_base}/product_profit"
)

In [65]:
local_output = "file:///Users/bhanupendyala/Downloads/exotic-indian-eats-pyspark/output"

In [66]:
monthly_sales_df.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(f"{local_output}/monthly_sales")

platform_sales_df.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(f"{local_output}/platform_sales")

product_profit_df.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(f"{local_output}/product_profit")

peak_hours_df.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(f"{local_output}/peak_hours")